
# Quantum Acoustic Qubit Stabilization — Full Pipeline (QuTiP + Differential Evolution)
This notebook:
1. Installs QuTiP.
2. Defines a one-qubit + one-acoustic-mode model with coherent drive **amplitude** and **phase** and engineered dissipation.
3. Computes steady-state metrics (purity, ground/excited populations, and fidelity to the target superposition \(|+\rangle\)).
4. Runs a coarse sweep for quick intuition.
5. Runs a differential-evolution optimizer to **maximize \(F_{+}=\langle +|\rho_q^{(ss)}|+\rangle\)** with purity regularization.
6. Saves CSVs and plots.


In [ ]:
# Install deps (Colab-friendly)
!pip -q install qutip numpy matplotlib scipy


In [ ]:

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
import os, csv, json
from scipy.optimize import differential_evolution

# ---------------------- Output dir ----------------------
OUT_DIR = "qutip_full_results"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------- Physical constants ----------------------
omega_q = 5.0 * 2*np.pi   # qubit frequency (GHz·2π)
omega_a = 2.0 * 2*np.pi   # acoustic mode (GHz·2π), used for thermal occupancy
N_a = 12                  # truncation of acoustic Hilbert space (tweak as needed)

gamma1 = 1.0e-5 * 2*np.pi   # qubit relaxation
gamma_phi = 1.0e-5 * 2*np.pi # qubit pure dephasing
kappa_nat = 1.0e-4 * 2*np.pi # natural acoustic decay

# Temperature and thermal occupancy
k_B = 1.380649e-23
h = 6.62607015e-34
GHz_to_Hz = 1e9
T_K = 0.015

def n_thermal(freq_GHz, T_K):
    f_hz = freq_GHz * GHz_to_Hz
    x = (h*f_hz)/(k_B*T_K + 1e-300)
    if x > 50:
        return 0.0
    return 1.0/(np.exp(x)-1.0)

n_th = n_thermal(omega_a/(2*np.pi), T_K)

# ---------------------- Operators ----------------------
Iq = qt.qeye(2); Ia = qt.qeye(N_a)
sx = qt.sigmax(); sz = qt.sigmaz(); sm = qt.sigmam()
a = qt.destroy(N_a); a_d = a.dag()

sxF = qt.tensor(sx, Ia)
szF = qt.tensor(sz, Ia)
smF = qt.tensor(sm, Ia)
aF  = qt.tensor(Iq, a)
aFd = aF.dag()

# ---------------------- Hamiltonian & Dissipators ----------------------
# H = 0.5*ωq σz + Δa a†a + 0.5*g σx (a + a†) + ε*(cosφ (a+a†) + sinφ i(a - a†))
# where Δa = ωa - ωd (we work in drive frame for the mode)
def build_H(g, eps, delta_a, phi):
    H0 = 0.5*omega_q*szF + delta_a*(aFd*aF) + 0.5*g*(sxF*(aF + aFd))
    # Quadrature-parametrized drive with phase φ
    X = (aF + aFd)           # position-like
    Y = 1j*(aF - aFd)        # momentum-like
    H_drive = eps * (np.cos(phi)*X + np.sin(phi)*Y)
    return H0 + H_drive

def build_cops(kappa_eng):
    k_tot = kappa_nat + kappa_eng
    C = []
    if gamma1 > 0: C.append(np.sqrt(gamma1)*smF)
    if gamma_phi > 0: C.append(np.sqrt(gamma_phi)*szF)
    if k_tot > 0:
        C.append(np.sqrt(k_tot*(n_th+1))*aF)
        if n_th > 0: C.append(np.sqrt(k_tot*n_th)*aFd)
    return C

# ---------------------- Metrics ----------------------
# Return purity, P_g, P_e, F_plus, rho_q
ket0 = qt.basis(2,0); ket1 = qt.basis(2,1)
ket_plus = (ket0 + ket1).unit()

def metrics(g, eps, delta_a, kappa_eng, phi):
    H = build_H(g, eps, delta_a, phi)
    C = build_cops(kappa_eng)
    rho = qt.steadystate(H, C, method='direct')
    rho_q = rho.ptrace(0)
    purity = float((rho_q*rho_q).tr().real)
    P_g = float(rho_q[1,1].real)  # ground if sz diag [+1,-1]
    P_e = float(rho_q[0,0].real)
    F_plus = float((ket_plus.dag() * rho_q * ket_plus).full().item())
    return purity, P_g, P_e, F_plus, rho_q

# Unit converters
toGHz = lambda MHz: MHz*1e-3*2*np.pi
toGHz_kHz = lambda kHz: kHz*1e-6*2*np.pi
eps_scale = 1e-4*2*np.pi  # drive scale; drive_units * eps_scale = ε


## Coarse Sweep (intuition building)

In [ ]:

g_list_MHz = np.array([0.3, 0.6, 1.2])
eps_units   = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
delta_MHz  = np.array([-25.0, -10.0, 0.0, 10.0, 25.0])
kappa_eng_kHz = np.array([0.0, 50.0, 200.0, 400.0])
phi_list   = np.array([0.0, np.pi/2, np.pi, 3*np.pi/2])

rows = []
best = {"F_plus": -1, "params": None, "purity": None}
for gM in g_list_MHz:
    for eu in eps_units:
        for dM in delta_MHz:
            for kk in kappa_eng_kHz:
                for phi in phi_list:
                    purity, Pg, Pe, Fp, _ = metrics(toGHz(gM), eu*eps_scale, toGHz(dM), toGHz_kHz(kk), phi)
                    rows.append([gM, eu, dM, kk, phi, purity, Pg, Pe, Fp])
                    if Fp > best["F_plus"]:
                        best = {"F_plus": Fp, "params": (gM, eu, dM, kk, phi), "purity": purity}

with open(os.path.join(OUT_DIR, "sweep_fidelity.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["g_MHz","drive_units","delta_MHz","kappa_eng_kHz","phi","purity","P_g","P_e","F_plus"])
    for r in rows:
        w.writerow([f"{r[0]:.3f}", f"{r[1]:.3f}", f"{r[2]:.3f}", f"{r[3]:.3f}", f"{r[4]:.6f}", f"{r[5]:.6f}", f"{r[6]:.6f}", f"{r[7]:.6f}", f"{r[8]:.6f}"])

print("Best sweep F_plus:", best["F_plus"])
print("Params [g_MHz, drive_units, delta_MHz, kappa_eng_kHz, phi]:", best["params"])
print("Purity at best F_plus:", best["purity"])


In [ ]:

# Visualization: F_plus vs (drive, detuning) at fixed g and kappa_eng; φ = 0 and φ = π/2
g_fix = 0.6
kk_fix = 200.0
phis = [0.0, np.pi/2]
for phi in phis:
    F = np.zeros((len(eps_units), len(delta_MHz)))
    for i, eu in enumerate(eps_units):
        for j, dM in enumerate(delta_MHz):
            _,_,_,Fp,_ = metrics(toGHz(g_fix), eu*eps_scale, toGHz(dM), toGHz_kHz(kk_fix), phi)
            F[i,j] = Fp
    plt.figure(figsize=(6,4))
    plt.imshow(F, origin='lower', aspect='auto', extent=[delta_MHz[0], delta_MHz[-1], 0, len(eps_units)-1])
    plt.xlabel("Δ_a (MHz)"); plt.ylabel("Drive index (ε / 1e-4·2π)")
    plt.title(f"F(|+>) heatmap — g≈{g_fix} MHz, κ_eng≈{kk_fix} kHz, φ≈{phi:.2f} rad")
    plt.colorbar(label="F(|+>)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"heatmap_Fplus_phi_{phi:.2f}.png"), dpi=200)
    plt.close()
print("Saved heatmaps for φ in", phis)


## Differential Evolution Optimization (Fidelity to |+⟩ + Purity Regularizer)

In [ ]:

# Objective: minimize J = 1 - F_plus + λ*(1 - purity)
lam = 0.1

def objective(x):
    gM, eu, dM, kk, phi = x
    purity, Pg, Pe, Fp, _ = metrics(toGHz(gM), eu*eps_scale, toGHz(dM), toGHz_kHz(kk), phi)
    J = 1.0 - Fp + lam*(1.0 - purity)
    return J

bounds = [
    (0.2, 1.5),   # g_MHz
    (0.0, 2.5),   # drive_units
    (-30.0, 30.0),# delta_MHz
    (0.0, 600.0), # kappa_eng_kHz
    (0.0, 2*np.pi) # phi
]

opt = differential_evolution(objective, bounds, maxiter=30, popsize=14, tol=1e-4, seed=7, polish=True)
gM_opt, eu_opt, dM_opt, kk_opt, phi_opt = opt.x
pur_opt, Pg_opt, Pe_opt, Fp_opt, rhoq_opt = metrics(toGHz(gM_opt), eu_opt*eps_scale, toGHz(dM_opt), toGHz_kHz(kk_opt), phi_opt)

opt_summary = {
    "optimum": {
        "g_MHz": float(gM_opt),
        "drive_units": float(eu_opt),
        "delta_MHz": float(dM_opt),
        "kappa_eng_kHz": float(kk_opt),
        "phi_rad": float(phi_opt),
        "F_plus": float(Fp_opt),
        "purity": float(pur_opt),
        "P_g": float(Pg_opt),
        "P_e": float(Pe_opt)
    }
}

with open(os.path.join(OUT_DIR, "opt_summary.json"), "w") as f:
    json.dump(opt_summary, f, indent=2)

print("Optimization complete.")
print("Optimum:", opt_summary["optimum"])


In [ ]:

# Plot F_plus vs phi at optimized (g, eps, delta, kappa_eng)
phis = np.linspace(0, 2*np.pi, 61)
Fphi = []
for p in phis:
    _,_,_,Fp,_ = metrics(toGHz({gM}), {eu}*eps_scale, toGHz({dM}), toGHz_kHz({kk}), p)

# NOTE: placeholders will be formatted later once we know the optimized values

In [ ]:

# Now that we have optimized parameters, regenerate F(phi)
with open(os.path.join(OUT_DIR, "opt_summary.json")) as f:
    opt = json.load(f)["optimum"]

gM = opt["g_MHz"]; eu = opt["drive_units"]; dM = opt["delta_MHz"]; kk = opt["kappa_eng_kHz"]
phis = np.linspace(0, 2*np.pi, 121)
Fphi = []
for p in phis:
    _,_,_,Fp,_ = metrics(toGHz(gM), eu*eps_scale, toGHz(dM), toGHz_kHz(kk), p)
    Fphi.append(Fp)

plt.figure(figsize=(6,4))
plt.plot(phis, Fphi)
plt.xlabel("φ (rad)"); plt.ylabel("F(|+>)")
plt.title("Fidelity vs Drive Phase at Optimized Params")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "Fplus_vs_phi_optimized.png"), dpi=200)
plt.close()

print("Saved:", os.path.join(OUT_DIR, "Fplus_vs_phi_optimized.png"))



## Exports
- `sweep_fidelity.csv` — coarse parameter sweep including φ.
- `heatmap_Fplus_phi_*.png` — heatmaps at two φ values.
- `opt_summary.json` — best parameters and resulting metrics.
- `Fplus_vs_phi_optimized.png` — phase dependence at the optimum.
